# Machine Learning for Applied Research

## 8: Hyperparameter Tuning

In this notebook, we will cover:
- Meeting a new dataset: Bike Sharing Demand
- Grid search: exhaustive, systematic search over hyperparameter combinations
- Random search: sampling combinations instead of trying all of them
- Reproducibility in research projects, now that tuning is in the picture

### Introduction

So far we have manually tried a handful of settings (tree depth, number of trees) and inspected the results ourselves. That does not scale once several hyperparameters interact with each other. This notebook covers systematic ways to search over hyperparameters, using the cross-validation machinery from Notebook 7 as the scoring mechanism, and introduces a new, larger dataset well suited to it.

### Meet a New Dataset: Bike Sharing Demand

Our dataset for this notebook is hourly bike rental counts from Washington DC's Capital Bikeshare system across 2011-2012, about 17,400 rows, considerably more than Energy Efficiency's 768. The larger size makes it a better fit for hyperparameter search, which requires fitting many models.

In [1]:
!pip install ucimlrepo --quiet

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

#### 1.1 Exercises:
> (a) Fetch the Bike Sharing dataset using `fetch_ucirepo(id=275)`.
>
> (b) Print the feature and target column names.
>
> (c) Build a working dataframe `bike_df` using only these columns as features: `season`, `yr`, `mnth`, `hr`, `holiday`, `weekday`, `workingday`, `weathersit`, `temp`, `atemp`, `hum`, `windspeed`, with `cnt` as the target.
>
> (d) Print the shape of `bike_df`.
>
> (e) Split the dataset into `X_bike_train` etc.

In [9]:
# Your solution:
# (a) Fetch the Bike Sharing dataset using `fetch_ucirepo(id=275)`.

bike_sharing = fetch_ucirepo(id=275)

# (b) Print the feature and target column names.
print("Feature columns:", bike_sharing.data.features.columns)
print(f"Target columns:  {bike_sharing.data.targets.columns.tolist()}")

# (c) Build a working dataframe `bike_df` using only these columns as features: `season`, `yr`, `mnth`, `hr`, `holiday`, `weekday`, `workingday`, `weathersit`, `temp`, `atemp`, `hum`, `windspeed`, with `cnt` as the target.
bike_df = bike_sharing.data.features[['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']].copy()
bike_df['cnt'] = bike_sharing.data.targets['cnt']

# (d) Print the shape of `bike_df`.
print("Shape of bike_df:", bike_df.shape)

# (e) Split the dataset into `X_bike_train` etc.
X_bike_train, X_bike_test, y_bike_train, y_bike_test = train_test_split(bike_df.drop('cnt', axis=1), bike_df['cnt'], test_size=0.25, random_state=42)

print(f"Train: {X_bike_train.shape[0]} rows | Test: {X_bike_test.shape[0]} rows")

Feature columns: Index(['dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
       'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed'],
      dtype='str')
Target columns:  ['cnt']
Shape of bike_df: (17379, 13)
Train: 13034 rows | Test: 4345 rows


##### Solution:

In [10]:
bikes = fetch_ucirepo(id=275)
print(f"Feature columns: {bikes.data.features.columns.tolist()}")
print(f"Target columns:  {bikes.data.targets.columns.tolist()}")

feature_cols = ["season", "yr", "mnth", "hr", "holiday", "weekday",
                 "workingday", "weathersit", "temp", "atemp", "hum", "windspeed"]

bike_df = pd.concat([bikes.data.features[feature_cols], bikes.data.targets[["cnt"]]], axis=1)

print(f"\nShape: {bike_df.shape}")

X_bike_train, X_bike_test, y_bike_train, y_bike_test = train_test_split(
    bike_df.drop(columns=["cnt"]), bike_df["cnt"], test_size=0.25, random_state=42
)
print(f"Train: {X_bike_train.shape[0]} rows | Test: {X_bike_test.shape[0]} rows")

Feature columns: ['dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed']
Target columns:  ['cnt']

Shape: (17379, 13)
Train: 13034 rows | Test: 4345 rows


### Grid Search: Exhaustive Systematic Search

Grid search defines a "grid" of candidate values for each hyperparameter, tries every combination, scores each one using cross-validation, and returns the combination with the best average score. It is thorough, but the number of combinations grows multiplicatively: 3 hyperparameters with 5 candidate values each is already 125 combinations, and with 5-fold cross-validation that means 625 separate model fits.

In [11]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [4, 6, 8, 10],
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
grid_search.fit(X_bike_train, y_bike_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV RMSE: {-grid_search.best_score_:.2f}")

Best parameters: {'max_depth': 10, 'n_estimators': 200}
Best CV RMSE: 51.86


#### 1.2 Exercises:
> (a) Fit a default `RandomForestRegressor(random_state=42)` (no tuning at all) on `X_bike_train`, `y_bike_train`.
>
> (b) Compute the test RMSE for both the default model and `grid_search.best_estimator_`.
>
> (c) How much did tuning improve the test RMSE? Was it worth the extra computation, in your judgment?

In [ ]:
# Your solution:
# (a) Fit a default `RandomForestRegressor(random_state=42)` (no tuning at all) on `X_bike_train`, `y_bike_train`.

default_rf = RandomForestRegressor(random_state=42)
default_rf.fit(X_bike_train, y_bike_train)

# (b) Compute the test RMSE for both the default model and `grid_search.best_estimator_`.
default_rmse = rmse(y_bike_test, default_rf.predict(X_bike_test))
tuned_rmse = rmse(y_bike_test, grid_search.best_estimator_.predict(X_bike_test))
print(f"Default model RMSE: {default_rmse:.2f}")
print(f"Tuned model RMSE: {tuned_rmse:.2f}")

# (c) How much did tuning improve the test RMSE? Was it worth the extra computation, in your judgment?
improvement = default_rmse - tuned_rmse
print(f"Improvement in RMSE: {improvement:.2f}")
# In my judgment, the improvement in RMSE is significant enough to justify the extra computation for tuning, especially if the model will be used in a production environment where accuracy is critical.

Default model RMSE: 42.97
Tuned model RMSE: 50.44


##### Solution:


In [13]:
default_forest = RandomForestRegressor(random_state=42)
default_forest.fit(X_bike_train, y_bike_train)

default_test_rmse = rmse(y_bike_test, default_forest.predict(X_bike_test))
tuned_test_rmse = rmse(y_bike_test, grid_search.best_estimator_.predict(X_bike_test))

print(f"Default Random Forest - Test RMSE: {default_test_rmse:.2f}")
print(f"Tuned Random Forest   - Test RMSE: {tuned_test_rmse:.2f}")

# (c) The improvement from tuning is often fairly modest for Random
# Forest specifically, since its defaults are already reasonably good,
# unlike some other model families, where default settings can be quite
# far from optimal. Whether the extra computation is "worth it" depends
# on context: for a one-off analysis, maybe not; for a model you will
# reuse or deploy many times, even a small, reliable improvement can
# justify a one-time tuning cost.

Default Random Forest - Test RMSE: 42.97
Tuned Random Forest   - Test RMSE: 50.44


#### 1.3 Exercises:
> (a) `grid_search.cv_results_` is a dictionary holding the full results of every combination tried, not just the best one. Convert it to a dataframe with `pd.DataFrame(grid_search.cv_results_)`.
>
> (b) From that dataframe, select just the `param_n_estimators`, `param_max_depth`, and `mean_test_score` columns, and sort by `mean_test_score` descending.
>
> (c) Looking at the sorted table, does performance change more when `n_estimators` varies or when `max_depth` varies? What would that suggest about which hyperparameter to prioritize if you only had time to tune one?

In [ ]:
# Your solution:
# (a) `grid_search.cv_results_` is a dictionary holding the full results of every combination tried, not just the best one. Convert it to a dataframe with `pd.DataFrame(grid_search.cv_results_)`.
cv_results_df = pd.DataFrame(grid_search.cv_results_)

# (b) From that dataframe, select just the `param_n_estimators`, `param_max_depth`, and `mean_test_score` columns, and sort by `mean_test_score` descending.
cv_results_subset = cv_results_df[['param_n_estimators', 'param_max_depth', 'mean_test_score']].sort_values('mean_test_score', ascending=False)
print(cv_results_subset)

# (c) Looking at the sorted table, does performance change more when `n_estimators` varies or when `max_depth` varies? What would that suggest about which hyperparameter to prioritize if you only had time to tune one?
# max_depth seems to have a more significant impact on performance than n_estimators, as the mean_test_score varies more noticeably with changes in max_depth. This suggests that if you only had time to tune one hyperparameter, it would be more beneficial to prioritize tuning max_depth over n_estimators.


    param_n_estimators  param_max_depth  mean_test_score
11                 200               10       -51.858639
10                 100               10       -52.059357
9                   50               10       -52.210843
8                  200                8       -65.122259
6                   50                8       -65.158220
7                  100                8       -65.229414
3                   50                6       -98.019521
5                  200                6       -98.076863
4                  100                6       -98.098025
2                  200                4      -117.545847
0                   50                4      -117.636733
1                  100                4      -117.642022


##### Solution:

In [17]:
results_df = pd.DataFrame(grid_search.cv_results_)
summary_df = results_df[["param_n_estimators", "param_max_depth", "mean_test_score"]].sort_values(
    "mean_test_score", ascending=False
)
print(summary_df)

# (c) In this search, mean_test_score usually shifts more as max_depth
# changes than as n_estimators changes at a fixed depth, depth controls
# how much each individual tree can overfit or underfit, while adding
# more trees mainly reduces variance from averaging (Notebook 4), which
# has diminishing returns past a point. If time were limited, tuning
# max_depth carefully and leaving n_estimators at a reasonably large
# fixed value (e.g. 200) would likely capture most of the benefit.

    param_n_estimators  param_max_depth  mean_test_score
11                 200               10       -51.858639
10                 100               10       -52.059357
9                   50               10       -52.210843
8                  200                8       -65.122259
6                   50                8       -65.158220
7                  100                8       -65.229414
3                   50                6       -98.019521
5                  200                6       -98.076863
4                  100                6       -98.098025
2                  200                4      -117.545847
0                   50                4      -117.636733
1                  100                4      -117.642022


### Random Search: Sampling Instead of Exhausting

Random search samples a fixed number of random hyperparameter combinations instead of trying every one. This might sound like a worse idea than being exhaustive, but it has two real advantages: it scales to many hyperparameters (or continuous ranges) without the combinatorial explosion of a grid, and in practice, usually only one or two hyperparameters matter much for a given model and dataset. So exhaustively covering every combination of the *unimportant* ones is largely wasted effort.

In [18]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distributions = {
    "n_estimators": randint(50, 300),
    "max_depth": randint(3, 15),
    "min_samples_leaf": randint(1, 10),
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_distributions,
    n_iter=15,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
)
random_search.fit(X_bike_train, y_bike_train)

print(f"Best parameters: {random_search.best_params_}")
print(f"Best CV RMSE: {-random_search.best_score_:.2f}")

Best parameters: {'max_depth': 13, 'min_samples_leaf': 3, 'n_estimators': 157}
Best CV RMSE: 45.95


Notice that `RandomizedSearchCV` itself takes a `random_state` - its sampling is random, so without pinning it down, rerunning the same code could search a different set of combinations and land on a different answer.

#### 1.4 Exercises:
> (a) How many total model fits did `grid_search` perform (combinations in `param_grid` × `cv` folds)? How many did `random_search` perform (`n_iter` × `cv` folds)?
>
> (b) Compare `grid_search.best_score_` and `random_search.best_score_`. Did allowing a third hyperparameter (`min_samples_leaf`) and continuous ranges help, hurt, or make little difference here?
>
> (c) In your own research, if you had 5 hyperparameters to tune instead of 2-3, would you reach for grid search or random search first? Why?

In [ ]:
# Your solution:
# (a) How many total model fits did `grid_search` perform (combinations in `param_grid` × `cv` folds)? How many did `random_search` perform (`n_iter` × `cv` folds)?
grid_search_fits = len(param_grid['n_estimators']) * len(param_grid['max_depth']) * 5  # 5-fold CV
random_search_fits = 15 * 5  # n_iter * cv

print(f"Grid search fits: {grid_search_fits}")
print(f"Random search fits: {random_search_fits}")

# (b) Compare `grid_search.best_score_` and `random_search.best_score_`. Did allowing a third hyperparameter (`min_samples_leaf`) and continuous ranges help, hurt, or make little difference here?
print(f"Grid search best score: {grid_search.best_score_:.2f}")
print(f"Random search best score: {random_search.best_score_:.2f}")
# Allowing a third hyperparameter and continuous ranges seems to have helped, as the random search achieved a better score than the grid search.

# (c) In your own research, if you had 5 hyperparameters to tune instead of 2-3, would you reach for grid search or random search first? Why?
# If I had 5 hyperparameters to tune, I would likely reach for random search first. This is because the number of combinations in a grid search grows exponentially with the number of hyperparameters and their possible values. Random search, on the other hand, can explore a larger portion of the hyperparameter space with fewer computational resources, making it more efficient for tuning a large number of hyperparameters.

Grid search fits: 60
Random search fits: 75
Grid search best score: -51.86
Random search best score: -45.95


##### Solution:


In [21]:
grid_fits = len(param_grid["n_estimators"]) * len(param_grid["max_depth"]) * 5
random_fits = 15 * 5

print(f"Grid search total fits:   {grid_fits}")
print(f"Random search total fits: {random_fits}")

print(f"\nGrid search best CV RMSE:   {-grid_search.best_score_:.2f}")
print(f"Random search best CV RMSE: {-random_search.best_score_:.2f}")

# (c) With 5 hyperparameters, an exhaustive grid becomes expensive very
# fast, even a modest 5 values per hyperparameter means 5**5 = 3,125
# combinations, each requiring a full cross-validation run. Random
# search's cost is controlled directly by n_iter regardless of how many
# hyperparameters or how wide their ranges are, which is why it tends to
# be the more practical default once you have more than 2-3 settings to
# tune at once.

Grid search total fits:   60
Random search total fits: 75

Grid search best CV RMSE:   51.86
Random search best CV RMSE: 45.95


#### 1.5 Exercises:
> (a) Rerun `RandomizedSearchCV` with the same `param_distributions`, but with `n_iter=5` and then `n_iter=40` (keep `cv=5`, `random_state=42`, `scoring` the same).
>
> (b) Compare `best_score_` across `n_iter=5`, `n_iter=15` (the original), and `n_iter=40`.
>
> (c) Does the improvement from 5 to 15 look similar in size to the improvement from 15 to 40? What does that suggest about picking `n_iter` in your own work, where compute time is limited?

In [24]:
# Your solution:
# (a) Rerun `RandomizedSearchCV` with the same `param_distributions`, but with `n_iter=5` and then `n_iter=40` (keep `cv=5`, `random_state=42`, `scoring` the same).

for n_iter in [5, 40]:
    random_search = RandomizedSearchCV(
        RandomForestRegressor(random_state=42),
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=5,
        scoring="neg_root_mean_squared_error",
        random_state=42,
        n_jobs=-1,
    )
    random_search.fit(X_bike_train, y_bike_train)
    print(f"n_iter={n_iter}: Best CV RMSE: {-random_search.best_score_:.2f}, Best parameters: {random_search.best_params_}")

# (b) Compare `best_score_` across `n_iter=5`, `n_iter=15` (the original), and `n_iter=40`.
print(f"n_iter=15: Best CV RMSE: {-random_search.best_score_:.2f}")

# (c) Does the improvement from 5 to 15 look similar in size to the improvement from 15 to 40? What does that suggest about picking `n_iter` in your own work, where compute time is limited?
# The improvement from 5 to 15 is more significant than the improvement from 15 to 40, suggesting that there are diminishing returns as `n_iter` increases. This indicates that in practice, one might choose a moderate value for `n_iter` (like 15) to balance between performance and computation time, especially when resources are limited.


n_iter=5: Best CV RMSE: 49.92, Best parameters: {'max_depth': 13, 'min_samples_leaf': 8, 'n_estimators': 238}
n_iter=40: Best CV RMSE: 45.95, Best parameters: {'max_depth': 13, 'min_samples_leaf': 3, 'n_estimators': 157}
n_iter=15: Best CV RMSE: 45.95


##### Solution:


In [23]:
for n_iter in [5, 40]:
    search_n = RandomizedSearchCV(
        RandomForestRegressor(random_state=42),
        param_distributions=param_distributions,
        n_iter=n_iter, cv=5, scoring="neg_root_mean_squared_error",
        random_state=42, n_jobs=-1,
    )
    search_n.fit(X_bike_train, y_bike_train)
    print(f"n_iter={n_iter:3d} - Best CV RMSE: {-search_n.best_score_:.2f}")

print(f"n_iter=15 (original) - Best CV RMSE: {-random_search.best_score_:.2f}")

# (c) The jump from n_iter=5 to n_iter=15 usually buys more improvement
# than the jump from 15 to 40 (diminishing returns).
# In practice this argues for starting with a modest n_iter budget 
# and only increasing it if the best score is still visibly 
# improving, rather than assuming "more search iterations" 
# always justifies its extra compute cost.

n_iter=  5 - Best CV RMSE: 49.92
n_iter= 40 - Best CV RMSE: 45.95
n_iter=15 (original) - Best CV RMSE: 45.95


### Reproducibility in Research Projects

With hyperparameter tuning in the toolbox, reproducibility becomes more important, not less! There are simply more places for randomness to hide:

- **Pin down every stochastic component**: the train/test split, the model itself (a Random Forest's bootstrap sampling), and the search procedure's own sampling (`RandomizedSearchCV`'s `random_state`). We have been doing this throughout, but tuning multiplies how many settings this applies to.
- **Report what was actually searched**: the exact grid or distributions tried, the cross-validation scheme (number of folds, whether stratified), the scoring metric, and the best settings found. "We tuned the model" is not reproducible, but the specifics are.
- **Note your library versions**: default hyperparameter values and even algorithm behavior can change between scikit-learn releases, so recording the version you used matters for anyone trying to reproduce your results exactly.

One subtlety: if you tune a model's hyperparameters using cross-validation and then report that same cross-validated score as your final result, is that a fair, unbiased estimate of how the model will perform on genuinely new data? Consider that the search already used every fold's data to help pick the winning settings.

#### 1.6 Exercises (discussion):
> (a) If you were to write a "Methods" paragraph describing the tuning in this notebook for a paper, list every specific detail from this notebook that ought to appear in it (aim for at least 5 items).
>
> (b) In your own research, which hyperparameters (if any) have you tuned so far, and how did you choose the values you tried? Would your approach hold up to the reproducibility standard described above?

In [ ]:
# Your solution:




##### Solution:


In [ ]:
# (a) A reasonably complete list: the model family (RandomForestRegressor),
# the exact hyperparameter grid or distributions searched, the search
# method (grid vs. random) and n_iter if random, the cross-validation
# scheme (cv=5, whether stratified), the scoring metric
# (neg_root_mean_squared_error), every random_state used (split, model,
# search), the best hyperparameters found, and the final held-out test
# score (not the CV score used during search).

# (b) Discussion, no fixed answer. A useful gut check: could a labmate
# rerun your exact search from your notes alone and get the same
# "winning" settings? If not, something on the list above is probably
# missing from your notes.

#### Challenge:
> `random_search.best_score_` was used to select the winning hyperparameters, which means it is already a little optimistic as an estimate of real-world performance. Compute the test RMSE of `random_search.best_estimator_` on `X_bike_test`/`y_bike_test` - data that played no role in the search, and compare it to `-random_search.best_score_`. Which number would you report in a paper as your model's expected performance, and why?

In [25]:
# Your solution:
# `random_search.best_score_` was used to select the winning hyperparameters, which means it is already a little optimistic as an estimate of real-world performance. Compute the test RMSE of `random_search.best_estimator_` on `X_bike_test`/`y_bike_test` - data that played no role in the search, and compare it to `-random_search.best_score_`. Which number would you report in a paper as your model's expected performance, and why?

print(f"Best CV RMSE (optimistic): {-random_search.best_score_:.2f}")
test_rmse = rmse(y_bike_test, random_search.best_estimator_.predict(X_bike_test))
print(f"Test RMSE (unbiased): {test_rmse:.2f}")



Best CV RMSE (optimistic): 45.95
Test RMSE (unbiased): 44.20


##### Solution:


In [ ]:
random_search_test_rmse = rmse(y_bike_test, random_search.best_estimator_.predict(X_bike_test))

print(f"Random search CV RMSE (used for tuning):  {-random_search.best_score_:.2f}")
print(f"Random search test RMSE (truly held out): {random_search_test_rmse:.2f}")

# The test RMSE computed on data that played no role whatsoever in the
# hyperparameter search is the honest number to report. The CV score
# used during the search, even though it comes from cross-validation, was
# itself used to pick the winning settings out of many candidates, which
# biases it slightly optimistic. This is the reasoning behind
# nested cross-validation: an outer loop purely for honest evaluation, an
# inner loop purely for tuning, kept strictly separate.

## Conclusion

This notebook turned "trying a few settings by hand" into a systematic process:

- **Grid search**: exhaustive and thorough, but its cost grows multiplicatively with each additional hyperparameter; `cv_results_` holds the full picture, not just the winner
- **Random search**: samples a fixed budget of combinations, scales far better to many hyperparameters, and often finds settings just as good, with diminishing returns as `n_iter` grows, similar to adding more trees to a forest
- **Reproducibility**: every stochastic component needs a `random_state`, and the exact search specification needs to be reported, not just "we tuned it"
- **The tuning-vs-evaluation distinction**: the CV score used to pick hyperparameters is mildly optimistic; a genuinely held-out test score is the honest number to report